In [12]:
print('Hello world')

Hello world


In [3]:
!pip install sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [4]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb

In [5]:
df = pd.read_csv('Crop_recommendation.csv')

In [6]:
df.head(5)

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [7]:
df.shape

(2200, 8)

In [8]:
df.iterrows()

<generator object DataFrame.iterrows at 0x7d8384c5ece0>

In [18]:
len(df)

2200

In [19]:
# Convertingg structured rows to natural language

documents = []

for idx, row in df.iterrows():
  text = (
    f"{row['label']} grows well with "
    f"nitrogen level of {row['N']} kg/ha, "
    f"phosphorus level of {row['P']} kg/ha, "
    f"potassium level of {row['K']} kg/ha, "
    f"temperature around {row['temperature']:.2f} degree Celsius, "
    f"humidity of {row['humidity']:.2f}%, "
    f"soil pH of {row['ph']:.2f}, "
    f"and rainfall of {row['rainfall']:.2f} mm."
  )

  metadata = {
       "crop": row['label'],
        "N": float(row['N']),
        "P": float(row['P']),
        "K": float(row['K']),
        "temperature": float(row['temperature']),
        "humidity": float(row['humidity']),
        "ph": float(row['ph']),
        "rainfall": float(row['rainfall'])
  }

  documents.append({
    "text":text,
    "metadata": metadata
})


Structured CSV rows were converted into natural language descriptions because embedding models captures semantic meaning better from textual context than raw tabular values.

In [20]:
len(documents)

2200

In [21]:
documents[0]

{'text': 'rice grows well with nitrogen level of 90 kg/ha, phosphorus level of 42 kg/ha, potassium level of 43 kg/ha, temperature around 20.88 degree Celsius, humidity of 82.00%, soil pH of 6.50, and rainfall of 202.94 mm.',
 'metadata': {'crop': 'rice',
  'N': 90.0,
  'P': 42.0,
  'K': 43.0,
  'temperature': 20.87974371,
  'humidity': 82.00274423,
  'ph': 6.502985292000001,
  'rainfall': 202.9355362}}

In [11]:
# Aggregate Crop documents

crop_summary = df.groupby('label').mean()

In [13]:
crop_summary

,N,P,K,temperature,humidity,ph,rainfall
label,,,,,,,
apple,20.80,134.22,199.89,22.630942,92.333383,5.929663,112.654779
banana,100.23,82.01,50.05,27.376798,80.358123,5.983893,104.626980
blackgram,40.02,67.47,19.24,29.973340,65.118426,7.133952,67.884151
chickpea,40.09,67.79,79.92,18.872847,16.860439,7.336957,80.058977
coconut,21.98,16.93,30.59,27.409892,94.844272,5.976562,175.686646
coffee,101.20,28.74,29.94,25.540477,58.869846,6.790308,158.066295
cotton,117.77,46.24,19.56,23.988958,79.843474,6.912675,80.398043
grapes,23.18,132.53,200.11,23.849575,81.875228,6.025937,69.611829
jute,78.40,46.86,39.99,24.958376,79.639864,6.732778,174.792798


In [14]:
aggregate_docs = []

for crop, row in crop_summary.iterrows():

  text = f""" {crop} generally prefers:
    nitrogen around {row['N']:.2f} kg/ha,
    phosphorus around {row['P']:.2f} kg/ha,
    potassium around {row['K']:.2f} kg/ha,
    temperature around {row['temperature']:.2f} degree Celsius,
    humidity around {row['humidity']:.2f}%,
    soil pH around {row['ph']:.2f},
    and rainfall around {row['rainfall']:.2f} mm."""

  aggregate_docs.append({
      "text": text,
      "metadata": {
          "crop":crop,
          "document_type":"aggregate"
      }

    })

Both row-level and aggregate crop documents were used. Row-level chunks preserve fine-grained environmental conditions, while aggregate chunks capture generalized crop behavior patterns.

In [22]:
# Combining both

all_docs = documents + aggregate_docs
len(all_docs)

2222

In [23]:
# Loading Embedding Model

model = SentenceTransformer('all-MiniLM-L6-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
# Creating Embeddings

texts = [doc['text'] for doc in all_docs]

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

In [25]:
embeddings.shape

(2222, 384)

In [34]:
# Chroma DB collection

client = chromadb.PersistentClient(path="./chromadb_store")

collection = client.create_collection(
    name="crop_recommendation"
)

In [35]:
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[doc['metadata'] for doc in all_docs],
    ids = [str(i) for i in range(len(all_docs))]
)

# Added Documents to Vector DB (This stores - Text, embeddings, metadata ..Inside vector DB)

In [30]:
# Semantic retrieval testing

query = 'crop suitable for high rainfall and acidic soil'
query_embedding = model.encode([query])

In [32]:
# Searching vector DB

results = collection.query(
    query_embeddings = query_embedding.tolist(),
    n_results=3
)

In [33]:
for doc in results['documents'][0]:
  print(doc)
  print("-"*50)

rice grows well with nitrogen level of 75 kg/ha, phosphorus level of 54 kg/ha, potassium level of 36 kg/ha, temperature around 26.29 degree Celsius, humidity of 84.57%, soil pH of 7.02, and rainfall of 257.49 mm.
--------------------------------------------------
rice grows well with nitrogen level of 60 kg/ha, phosphorus level of 43 kg/ha, potassium level of 44 kg/ha, temperature around 21.02 degree Celsius, humidity of 82.95%, soil pH of 7.42, and rainfall of 298.40 mm.
--------------------------------------------------
rice grows well with nitrogen level of 72 kg/ha, phosphorus level of 40 kg/ha, potassium level of 38 kg/ha, temperature around 20.41 degree Celsius, humidity of 82.21%, soil pH of 7.59, and rainfall of 245.15 mm.
--------------------------------------------------


What Ingestion Pipeline did:


*   CSV Loading
*   Row to text Conversion
*   Chunking Strategy
*   Embedding Generation
*   Vector indexing
*   MetaData storage

















In [36]:
!zip -r chromadb_store.zip chromadb_store

  adding: chromadb_store/ (stored 0%)
  adding: chromadb_store/f77d2280-d893-47fc-a1cd-61fa0f0c3fc8/ (stored 0%)
  adding: chromadb_store/f77d2280-d893-47fc-a1cd-61fa0f0c3fc8/length.bin (deflated 79%)
  adding: chromadb_store/f77d2280-d893-47fc-a1cd-61fa0f0c3fc8/data_level0.bin (deflated 13%)
  adding: chromadb_store/f77d2280-d893-47fc-a1cd-61fa0f0c3fc8/link_lists.bin (deflated 86%)
  adding: chromadb_store/f77d2280-d893-47fc-a1cd-61fa0f0c3fc8/header.bin (deflated 57%)
  adding: chromadb_store/f77d2280-d893-47fc-a1cd-61fa0f0c3fc8/index_metadata.pickle (deflated 63%)
  adding: chromadb_store/chroma.sqlite3 (deflated 53%)
